In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Ajout du src au path
sys.path.insert(0, '../src')

# ============================================================
# CONTournement du __init__.py problématique
# ============================================================
# Crée un module vide pour xai_clinical afin d'éviter l'import
# de pipeline.py qui appelle sys.stdout.reconfigure()
import types
if 'xai_clinical' not in sys.modules:
    sys.modules['xai_clinical'] = types.ModuleType('xai_clinical')
    sys.modules['xai_clinical'].__path__ = []

# Crée les sous-modules nécessaires
for submod in ['xai_clinical.data', 'xai_clinical.models']:
    if submod not in sys.modules:
        sys.modules[submod] = types.ModuleType(submod)

# ============================================================
# Imports directs (sans passer par __init__.py)
# ============================================================
import importlib.util

# Chargement direct des fichiers
spec_loader = importlib.util.spec_from_file_location(
    "pancan_loader", 
    os.path.join(os.path.dirname(os.getcwd()), '../src/xai_clinical/data/pancan_loader.py')
)
module_loader = importlib.util.module_from_spec(spec_loader)
sys.modules['xai_clinical.data.pancan_loader'] = module_loader
spec_loader.loader.exec_module(module_loader)

spec_fusion = importlib.util.spec_from_file_location(
    "pancan_fusion",
    os.path.join(os.path.dirname(os.getcwd()), '../src/xai_clinical/data/pancan_fusion.py')
)
module_fusion = importlib.util.module_from_spec(spec_fusion)
sys.modules['xai_clinical.data.pancan_fusion'] = module_fusion
spec_fusion.loader.exec_module(module_fusion)

spec_model = importlib.util.spec_from_file_location(
    "pancan_model",
    os.path.join(os.path.dirname(os.getcwd()), '../src/xai_clinical/models/pancan_model.py')
)
module_model = importlib.util.module_from_spec(spec_model)
sys.modules['xai_clinical.models.pancan_model'] = module_model
spec_model.loader.exec_module(module_model)

# Imports depuis les modules chargés
PANCANLoader = module_loader.PANCANLoader
BRCALoader = module_loader.BRCALoader
PANCANBRCAFusion = module_loader.PANCANBRCAFusion
MultiOmicsFusion = module_fusion.MultiOmicsFusion
FeatureSelector = module_fusion.FeatureSelector
SurvivalTargetBuilder = module_fusion.SurvivalTargetBuilder
XAIPancanModel = module_model.XAIPancanModel
GenePathwayAnalyzer = module_model.GenePathwayAnalyzer

print("✅ Configuration OK - Modules PANCAN chargés directement (sans __init__.py)")

# %% [markdown]
# ## 1. Chargement PANCAN (Expression Génique)

# %%
# Chargement PANCAN
pancan = PANCANLoader(PANCAN_DIR, cancer_type="BRCA")

print("Chargement EBPlusPlusAdjustPANCAN.tsv...")
pan_expr = pancan.load_gene_expression()
print(f"Expression PANCAN BRCA: {pan_expr.shape}")

print("\nPremiers échantillons:")
print(pan_expr.head(3).iloc[:, :5])

# %% [markdown]
# ## 2. Chargement BRCA TCGA (Clinique)

# %%
# Chargement BRCA
brca = BRCALoader(BRCA_DIR)

print("Données cliniques BRCA...")
brca_clin = brca.get_merged_clinical()
print(f"Clinical merged: {brca_clin.shape}")

print("\nColonnes cliniques disponibles:")
print(list(brca_clin.columns))

# %%
# Données omiques BRCA (optionnel)
brca_mrna = brca.load_mrna_seq()
brca_mut = brca.load_mutations()
brca_cna = brca.load_cna()

print(f"BRCA mRNA: {brca_mrna.shape if brca_mrna is not None else 'N/A'}")
print(f"BRCA Mutations: {brca_mut.shape if brca_mut is not None else 'N/A'}")
print(f"BRCA CNA: {brca_cna.shape if brca_cna is not None else 'N/A'}")

# %% [markdown]
# ## 3. Construction de la Cible (Survie)

# %%
# Extraction de la cible de survie
target = brca.extract_survival_target(brca_clin, cutoff_months=60)

# Distribution
plt.figure(figsize=(6, 4))
target.value_counts().plot(kind='bar', color=['#2ecc71', '#e74c3c'])
plt.title('Distribution de la Cible de Survie (60 mois)')
plt.xlabel('Classe (0=Survie, 1=Décès)')
plt.ylabel('Nombre de patients')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'target_distribution.png'), dpi=300, bbox_inches='tight')
plt.show()

print(f"\nDistribution de la cible:\n{target.value_counts()}")

# %% [markdown]
# ## 4. Prétraitement PANCAN

# %%
# Alignement et prétraitement
aligned = pancan.align_data()

expr_processed = pancan.preprocess_expression(
    aligned['expression'],
    min_variance=0.5,
    top_genes=5000
)

aligned['expression'] = expr_processed
pancan.save_processed(os.path.join(DATA_DIR, 'processed', 'pancan'), aligned)

print(f"Expression prétraitée: {expr_processed.shape}")

# %% [markdown]
# ## 5. Fusion PANCAN + BRCA

# %%
# Fusion
fusion = PANCANBRCAFusion(pancan, brca)
fused = fusion.fuse_expression_clinical(expr_processed, brca_clin, target)

if not fused.empty:
    print(f"Fusion réussie: {fused.shape}")
    print(f"\nAperçu:")
    print(fused.iloc[:3, :5])
else:
    print("⚠️ Fusion vide - problème d'IDs")

# %%
# Préparation ML
X, y = fusion.prepare_ml_features(fused)

print(f"X: {X.shape}, y: {y.value_counts().to_dict()}")

# %% [markdown]
# ## 6. Fusion Multi-Omiques

# %%
# Séparation expression / clinique
expr_cols = [c for c in X.columns if len(str(c)) > 3 and not str(c).startswith('clinical_')]
clin_cols = [c for c in X.columns if c not in expr_cols]

expression_df = X[expr_cols] if expr_cols else X.iloc[:, :X.shape[1]//2]
clinical_df = X[clin_cols] if clin_cols else X.iloc[:, X.shape[1]//2:]

print(f"Expression: {expression_df.shape}")
print(f"Clinical: {clinical_df.shape}")

# Fusion
multi_fusion = MultiOmicsFusion(
    expression_data=expression_df,
    clinical_data=clinical_df,
    fusion_strategy="early"
)

clin_features = multi_fusion.prepare_clinical_features()
expr_reduced = multi_fusion.reduce_expression_dimension(n_components=100)
X_fused = multi_fusion.fuse(clin_features, expr_reduced)

print(f"\nFeatures fusionnées: {X_fused.shape}")

# %% [markdown]
# ## 7. Modèle XAI

# %%
# Entraînement
model = XAIPancanModel(model_type="random_forest", random_state=42)
model.prepare_data(X_fused, y, test_size=0.2, scale=True)

print("Entraînement...")
model.train()

print("\nValidation croisée...")
cv = model.cross_validate(cv=5)
print(f"CV AUC: {cv['mean_auc']:.4f} ± {cv['std_auc']*2:.4f}")

# %% [markdown]
# ## 8. SHAP + Interprétation

# %%
# SHAP
print("Calcul SHAP...")
model.explain_with_shap(background_samples=100, test_samples=100)

gene_importance = model.get_gene_importance(top_n=50)
print("\nTop 20 features:")
print(gene_importance.head(20))

# %%
# Plot SHAP summary
model.plot_shap_summary(save_path=os.path.join(OUTPUT_DIR, 'shap_summary.png'))

# %%
# Plot importance des gènes
model.plot_gene_importance(top_n=30, save_path=os.path.join(OUTPUT_DIR, 'gene_importance.png'))

# %%
# Plot ROC
model.plot_roc_curve(save_path=os.path.join(OUTPUT_DIR, 'roc_curve.png'))

# %% [markdown]
# ## 9. Pathways

# %%
pathway_analyzer = GenePathwayAnalyzer(gene_importance)
pathways = pathway_analyzer.map_to_pathways()

if not pathways.empty:
    print("Pathways enrichis:")
    print(pathways)
    pathway_analyzer.plot_pathway_enrichment(save_path=os.path.join(OUTPUT_DIR, 'pathways.png'))
else:
    print("Aucun pathway enrichi trouvé")

# %% [markdown]
# ## 10. Sauvegarde

# %%
model.save_results(OUTPUT_DIR)
fused.to_csv(os.path.join(DATA_DIR, 'processed', 'pancan_brca_fused.csv'))

print(f"✅ Résultats dans {OUTPUT_DIR}")
print("\nFichiers générés:")
for f in os.listdir(OUTPUT_DIR):
    print(f"  - {f}")

